# Food101 Classification Benchmark

This notebook trains every backbone defined in `models.py` on the Food101 dataset with both `NoAug` and `Aug` pipelines, then exports the same core statistics structure used in the Oxford Pets reference page.

## What this notebook produces

- Full training for `resnet50`, `mobilenet_v3`, `vit_b16`, and `swin_b`
- Two experiment modes per backbone: `NoAug` and `Aug`
- Quick summary blocks: best model, performance range, fastest model
- Performance table: Precision, Recall, F1-Score, Accuracy, Top-5 Accuracy
- Resource table: Training Time, Inference Time, FLOPs, Parameters, Model Size
- Configuration tables for training, data, models, and evaluation
- Interactive learning curves, confusion matrices, and per-class reports
- Exported CSV/JSON artifacts in `results/food101_classification` for future web integration

The report layout is adapted from the Oxford Pets classification demo: [Classification - Oxford Pets](https://ltsach.github.io/AILearningHub/03_Computer_Vision/oxford_pets/classification/index.html).

In [1]:
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import torch
from IPython.display import Markdown, display

from models import list_backbones
from training_pipeline import (
    ExperimentConfig,
    build_quick_summary,
    build_report_tables,
    create_confusion_matrix_figure,
    create_learning_curve_figure,
    export_website_comparison_payload,
    fit_single_experiment,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:,.3f}".format

In [2]:
DEVICE = torch.device("cuda")

CONFIG = ExperimentConfig(
    batch_size=1024,
    num_workers=8,
    learning_rate=1e-3,
    max_epochs=40,
    patience=5,
    min_delta=1e-3,
    benchmark_runs=100,
    warmup_runs=10,
    results_root="results/food101_classification",
)

print(f"Available backbones: {list_backbones()}")
print(f"Results root: {Path(CONFIG.results_root)}")


Available backbones: ['resnet50', 'mobilenet_v3', 'vit_b16', 'swin_b']
Results root: results\food101_classification


## 1. Train Each Backbone

In [3]:
def run_backbone_experiments(backbone: str):
    return [
        fit_single_experiment(
            backbone=backbone,
            use_augmentation=False,
            config=CONFIG,
            device=torch.device("cuda"),
        ),
        fit_single_experiment(
            backbone=backbone,
            use_augmentation=True,
            config=CONFIG,
            device=torch.device("cuda"),
        ),
    ]

print("Run the backbone cells below one by one, then run the aggregate/report cells.")

Run the backbone cells below one by one, then run the aggregate/report cells.


In [4]:
resnet50_runs = run_backbone_experiments("resnet50")
display(pd.DataFrame([run["metadata"] for run in resnet50_runs])[
    ["experiment_name", "best_epoch", "training_time_minutes"]
])

e:\DL-assign1\image\.venv\Lib\site-packages\torch\utils\data\dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Dataset already exists !
Path to dataset files: ./data\food-101
Dataset already exists !
Path to dataset files: ./data\food-101


e:\DL-assign1\image\.venv\Lib\site-packages\torch\utils\data\dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


,experiment_name,best_epoch,training_time_minutes
0,ResNet50 (NoAug),15,13.791
1,ResNet50 (Aug),11,13.111


In [5]:
mobilenet_v3_runs = run_backbone_experiments("mobilenet_v3")
display(pd.DataFrame([run["metadata"] for run in mobilenet_v3_runs])[
    ["experiment_name", "best_epoch", "training_time_minutes"]
])

e:\DL-assign1\image\.venv\Lib\site-packages\torch\utils\data\dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Dataset already exists !
Path to dataset files: ./data\food-101
Dataset already exists !
Path to dataset files: ./data\food-101


e:\DL-assign1\image\.venv\Lib\site-packages\torch\utils\data\dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


,experiment_name,best_epoch,training_time_minutes
0,MobileNetV3 Large 100 (NoAug),9,8.179
1,MobileNetV3 Large 100 (Aug),11,12.488


In [6]:
vit_b16_runs = run_backbone_experiments("vit_b16")
display(pd.DataFrame([run["metadata"] for run in vit_b16_runs])[
    ["experiment_name", "best_epoch", "training_time_minutes"]
])

e:\DL-assign1\image\.venv\Lib\site-packages\torch\utils\data\dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Dataset already exists !
Path to dataset files: ./data\food-101
Dataset already exists !
Path to dataset files: ./data\food-101


e:\DL-assign1\image\.venv\Lib\site-packages\torch\utils\data\dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


,experiment_name,best_epoch,training_time_minutes
0,ViT-B/16 (NoAug),10,19.852
1,ViT-B/16 (Aug),8,17.866


In [7]:
swin_b_runs = run_backbone_experiments("swin_b")
display(pd.DataFrame([run["metadata"] for run in swin_b_runs])[
    ["experiment_name", "best_epoch", "training_time_minutes"]
])

e:\DL-assign1\image\.venv\Lib\site-packages\torch\utils\data\dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Dataset already exists !
Path to dataset files: ./data\food-101
Dataset already exists !
Path to dataset files: ./data\food-101


e:\DL-assign1\image\.venv\Lib\site-packages\torch\utils\data\dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


,experiment_name,best_epoch,training_time_minutes
0,Swin-B (NoAug),7,20.226
1,Swin-B (Aug),7,20.853


In [8]:
experiments = {}
for run_group in [
    resnet50_runs,
    mobilenet_v3_runs,
    vit_b16_runs,
    swin_b_runs,
]:
    for experiment in run_group:
        experiments[experiment["metadata"]["experiment_slug"]] = experiment

output_dir = Path(CONFIG.output_dir)
bundle = {
    "experiments": experiments,
    "output_dir": output_dir,
}
experiment_list = list(experiments.values())

tables = build_report_tables(experiments)
performance_df = tables["performance"]
resource_df = tables["resources"]
model_cfg_df = tables["model_config"]
quick_summary = build_quick_summary(performance_df, resource_df)

web_payload_path = output_dir / "model-classification-comparison-data.json"
web_payload = export_website_comparison_payload(experiments, web_payload_path)

sample_meta = experiment_list[0]["metadata"]
configuration_tables = {
    "training": pd.DataFrame(
        [
            {"Setting": "Mode", "Value": "Transfer Learning (Freeze Backbone)"},
            {"Setting": "Trainable Layers", "Value": "Classification head only"},
            {"Setting": "Optimizer", "Value": "Adam"},
            {"Setting": "Learning Rate", "Value": CONFIG.learning_rate},
            {"Setting": "Batch Size", "Value": CONFIG.batch_size},
            {"Setting": "Max Epochs", "Value": CONFIG.max_epochs},
            {
                "Setting": "Early Stopping",
                "Value": f"Patience = {CONFIG.patience}, Min Delta = {CONFIG.min_delta}",
            },
            {"Setting": "Loss Function", "Value": "Cross Entropy Loss"},
        ]
    ),
    "data": pd.DataFrame(
        [
            {"Setting": "Dataset", "Value": "Food101"},
            {"Setting": "Training Set", "Value": f"{sample_meta['split_sizes']['train']:,} images"},
            {"Setting": "Validation Set", "Value": f"{sample_meta['split_sizes']['val']:,} images"},
            {"Setting": "Test Set", "Value": f"{sample_meta['split_sizes']['test']:,} images"},
            {"Setting": "Number of Classes", "Value": len(sample_meta['class_names'])},
            {
                "Setting": "Input Size",
                "Value": (
                    f"{sample_meta['data_config']['input_size'][1]}"
                    f"x{sample_meta['data_config']['input_size'][2]}"
                    f"x{sample_meta['data_config']['input_size'][0]}"
                ),
            },
            {"Setting": "Normalization Mean", "Value": sample_meta["data_config"]["mean"]},
            {"Setting": "Normalization Std", "Value": sample_meta["data_config"]["std"]},
            {"Setting": "No Augmentation", "Value": "Resize -> CenterCrop -> ToTensor -> Normalize"},
            {
                "Setting": "With Augmentation",
                "Value": "Resize -> RandomResizedCrop -> RandomHorizontalFlip -> ColorJitter -> ToTensor -> Normalize",
            },
        ]
    ),
    "evaluation": pd.DataFrame(
        [
            {
                "Setting": "Metrics",
                "Value": "Accuracy, Precision, Recall, F1-Score, Top-5 Accuracy",
            },
            {"Setting": "Averaging Method", "Value": "Macro average"},
            {"Setting": "Training Hardware", "Value": torch.cuda.get_device_name(0)},
            {
                "Setting": "Inference Benchmark",
                "Value": f"Single image, average over {CONFIG.benchmark_runs} runs",
            },
        ]
    ),
}

print(f"Artifacts directory: {output_dir.resolve()}")
print(f"Experiments loaded: {len(experiments)}")
print(f"Web payload exported: {web_payload_path}")
print(f"Payload models: {len(web_payload['models'])}")
display(performance_df.head())

Artifacts directory: E:\DL-assign1\image\results\food101_classification
Experiments loaded: 8
Web payload exported: results\food101_classification\model-classification-comparison-data.json
Payload models: 8


,Model,Precision ↑,Recall ↑,F1-Score ↑,Accuracy ↑,Top-5 Acc ↑
0,Swin-B (NoAug),87.998,87.861,87.816,87.861,97.538
1,Swin-B (Aug),87.846,87.677,87.647,87.677,97.498
2,ViT-B/16 (Aug),83.480,83.307,83.275,83.307,95.960
3,ViT-B/16 (NoAug),83.450,83.129,83.149,83.129,95.901
4,ResNet50 (Aug),79.802,79.538,79.457,79.538,94.469


## 2. Quick Summary

In [9]:
summary_md = f"""
### Best Performing Model

- **{quick_summary['best_model']['name']}**
- **{quick_summary['best_model']['accuracy']:.2f}% Accuracy**
- **{quick_summary['best_model']['f1']:.2f}% F1-Score**

### Performance Range

- Accuracy: **{quick_summary['performance_range']['accuracy_min']:.2f}% - {quick_summary['performance_range']['accuracy_max']:.2f}%**
- F1-Score: **{quick_summary['performance_range']['f1_min']:.2f}% - {quick_summary['performance_range']['f1_max']:.2f}%**
- Top-5 Accuracy: **{quick_summary['performance_range']['top5_min']:.2f}% - {quick_summary['performance_range']['top5_max']:.2f}%**

### Fastest Model

- **{quick_summary['fastest_model']['name']}**
- **{quick_summary['fastest_model']['inference_ms']:.3f} ms / sample**
- **{quick_summary['fastest_model']['accuracy']:.2f}% Accuracy**
"""

display(Markdown(summary_md))

top_summary_df = performance_df[["Model", "Accuracy ↑", "F1-Score ↑"]].head(5).copy()
display(top_summary_df)


### Best Performing Model

- **Swin-B (NoAug)**
- **87.86% Accuracy**
- **87.82% F1-Score**

### Performance Range

- Accuracy: **55.23% - 87.86%**
- F1-Score: **54.61% - 87.82%**
- Top-5 Accuracy: **78.90% - 97.54%**

### Fastest Model

- **MobileNetV3 Large 100 (NoAug)**
- **1.651 ms / sample**
- **55.23% Accuracy**


,Model,Accuracy ↑,F1-Score ↑
0,Swin-B (NoAug),87.861,87.816
1,Swin-B (Aug),87.677,87.647
2,ViT-B/16 (Aug),83.307,83.275
3,ViT-B/16 (NoAug),83.129,83.149
4,ResNet50 (Aug),79.538,79.457


In [10]:
display(Markdown("## 3. Report Tables"))

## 3. Report Tables

In [11]:
display(Markdown("### Performance"))
display(
    performance_df.style.format(
        {
            "Precision ↑": "{:.2f}%",
            "Recall ↑": "{:.2f}%",
            "F1-Score ↑": "{:.2f}%",
            "Accuracy ↑": "{:.2f}%",
            "Top-5 Acc ↑": "{:.2f}%",
        }
    ).highlight_max(
        subset=["Precision ↑", "Recall ↑", "F1-Score ↑", "Accuracy ↑", "Top-5 Acc ↑"],
        color="#d8f3dc",
    )
)

display(Markdown("### Resources"))
display(
    resource_df.style.format(
        {
            "Training Time ↓ (min)": "{:.2f}",
            "Inference Time ↓ (ms)": "{:.3f}",
            "FLOPs ↓ (B)": "{:.3f}",
            "Parameters ↓ (M)": "{:.3f}",
            "Model Size ↓ (MB)": "{:.3f}",
        }
    ).highlight_min(
        subset=["Training Time ↓ (min)", "Inference Time ↓ (ms)", "FLOPs ↓ (B)", "Parameters ↓ (M)", "Model Size ↓ (MB)"],
        color="#fff3bf",
    )
)

display(Markdown("### Model Configurations"))
display(
    model_cfg_df.style.format(
        {
            "Parameters (M)": "{:.3f}",
            "Trainable Parameters (M)": "{:.3f}",
        }
    )
)

### Performance

,Model,Precision ↑,Recall ↑,F1-Score ↑,Accuracy ↑,Top-5 Acc ↑
0,Swin-B (NoAug),88.00%,87.86%,87.82%,87.86%,97.54%
1,Swin-B (Aug),87.85%,87.68%,87.65%,87.68%,97.50%
2,ViT-B/16 (Aug),83.48%,83.31%,83.28%,83.31%,95.96%
3,ViT-B/16 (NoAug),83.45%,83.13%,83.15%,83.13%,95.90%
4,ResNet50 (Aug),79.80%,79.54%,79.46%,79.54%,94.47%
5,ResNet50 (NoAug),79.52%,79.35%,79.27%,79.35%,94.61%
6,MobileNetV3 Large 100 (Aug),55.72%,55.98%,55.31%,55.98%,79.80%
7,MobileNetV3 Large 100 (NoAug),54.80%,55.23%,54.61%,55.23%,78.90%


### Resources

,Model,Training Time ↓ (min),Inference Time ↓ (ms),FLOPs ↓ (B),Parameters ↓ (M),Model Size ↓ (MB)
0,MobileNetV3 Large 100 (NoAug),8.18,1.651,0.432,4.912,19.065
1,MobileNetV3 Large 100 (Aug),12.49,1.656,0.432,4.912,19.065
2,ResNet50 (NoAug),13.79,2.796,0.002,24.605,94.051
3,ResNet50 (Aug),13.11,2.849,0.002,24.605,94.051
4,ViT-B/16 (NoAug),19.85,3.597,33.696,86.246,329.169
5,ViT-B/16 (Aug),17.87,3.600,33.696,86.246,329.169
6,Swin-B (NoAug),20.23,5.759,30.339,87.322,333.353
7,Swin-B (Aug),20.85,5.792,30.339,87.322,333.353


### Model Configurations

,Model,Backbone Key,Family,Pretrained Source,timm Name,Parameters (M),Trainable Parameters (M)
0,ResNet50,resnet50,cnn,BiT / ImageNet-21k,resnetv2_50x1_bit.goog_in21k,24.605,1.105
1,MobileNetV3 Large 100,mobilenet_v3,cnn,MIIL / ImageNet-21k,mobilenetv3_large_100.miil_in21k,4.912,0.710
2,ViT-B/16,vit_b16,transformer,ImageNet-21k,vit_base_patch16_224.orig_in21k,86.246,0.447
3,Swin-B,swin_b,transformer,ImageNet-22k,swin_base_patch4_window7_224.ms_in22k,87.322,0.579


## 4. Interactive Learning Curves

In [12]:
history_options = [
    (experiment["metadata"]["experiment_name"], slug)
    for slug, experiment in bundle["experiments"].items()
]

metric_options = [
    ("Accuracy", "accuracy"),
    ("Loss", "loss"),
    ("Precision", "precision"),
    ("Recall", "recall"),
    ("F1-Score", "f1"),
    ("Top-5 Accuracy", "top5_accuracy"),
]

history_dropdown = widgets.Dropdown(
    options=history_options,
    description="Model:",
    value=history_options[0][1],
)
metric_dropdown = widgets.Dropdown(
    options=metric_options,
    description="Metric:",
    value="accuracy",
)


def show_learning_curve(model_slug, metric):
    experiment = bundle["experiments"][model_slug]
    fig = create_learning_curve_figure(
        history_df=experiment["history"],
        experiment_name=experiment["metadata"]["experiment_name"],
        metric=metric,
    )
    fig.show()


display(widgets.HBox([history_dropdown, metric_dropdown]))
curve_output = widgets.interactive_output(
    show_learning_curve,
    {"model_slug": history_dropdown, "metric": metric_dropdown},
)
display(curve_output)

Output()

## 5. Interactive Confusion Matrices

In [13]:
confusion_dropdown = widgets.Dropdown(
    options=history_options,
    description="Model:",
    value=history_options[0][1],
)


def show_confusion_matrix(model_slug):
    experiment = bundle["experiments"][model_slug]
    fig = create_confusion_matrix_figure(
        confusion_df=experiment["confusion_matrix"],
        experiment_name=experiment["metadata"]["experiment_name"],
    )
    fig.show()


display(confusion_dropdown)
confusion_output = widgets.interactive_output(
    show_confusion_matrix,
    {"model_slug": confusion_dropdown},
)
display(confusion_output)

Dropdown(description='Model:', options=(('ResNet50 (NoAug)', 'resnet50_no_aug'), ('ResNet50 (Aug)', 'resnet50_…

Output()

## 6. Configuration Tables

In [14]:
display(Markdown("### Training Configuration"))
display(configuration_tables["training"])

display(Markdown("### Data Configuration"))
display(configuration_tables["data"])

display(Markdown("### Evaluation Configuration"))
display(configuration_tables["evaluation"])

### Training Configuration

,Setting,Value
0,Mode,Transfer Learning (Freeze Backbone)
1,Trainable Layers,Classification head only
2,Optimizer,Adam
3,Learning Rate,0.001
4,Batch Size,1024
5,Max Epochs,40
6,Early Stopping,"Patience = 5, Min Delta = 0.001"
7,Loss Function,Cross Entropy Loss


### Data Configuration

,Setting,Value
0,Dataset,Food101
1,Training Set,"70,700 images"
2,Validation Set,"15,150 images"
3,Test Set,"15,150 images"
4,Number of Classes,101
5,Input Size,224x224x3
6,Normalization Mean,"[0.5, 0.5, 0.5]"
7,Normalization Std,"[0.5, 0.5, 0.5]"
8,No Augmentation,Resize -> CenterCrop -> ToTensor -> Normalize
9,With Augmentation,Resize -> RandomResizedCrop -> RandomHorizonta...


### Evaluation Configuration

,Setting,Value
0,Metrics,"Accuracy, Precision, Recall, F1-Score, Top-5 A..."
1,Averaging Method,Macro average
2,Training Hardware,NVIDIA GeForce GTX 1660 SUPER
3,Inference Benchmark,"Single image, average over 100 runs"


## 7. Metrics Formulas

### Accuracy

\[
\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}
\]

### Precision

\[
\text{Precision} = \frac{TP}{TP + FP}
\]

### Recall

\[
\text{Recall} = \frac{TP}{TP + FN}
\]

### F1-Score

\[
\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
\]

### Top-5 Accuracy

\[
\text{Top-5 Accuracy} = \frac{\text{Number of correct predictions in top-5}}{\text{Total predictions}}
\]

All summary tables in this notebook use **macro averaging** for Precision, Recall, and F1-Score to stay consistent across classes.

In [15]:
artifact_rows = [
    {
        "name": path.name,
        "type": "directory" if path.is_dir() else "file",
    }
    for path in sorted(output_dir.glob("*"))
]

display(Markdown("## 8. Exported Artifacts"))
display(pd.DataFrame(artifact_rows))

## 8. Exported Artifacts

,name,type
0,mobilenet_v3_aug,directory
1,mobilenet_v3_no_aug,directory
2,model-classification-comparison-data.json,file
3,resnet50_aug,directory
4,resnet50_no_aug,directory
5,swin_b_aug,directory
6,swin_b_no_aug,directory
7,vit_b16_aug,directory
8,vit_b16_no_aug,directory


## 9. Per-Class Classification Report

The table below is saved for every experiment as `classification_report.csv`, which can be consumed directly by a future web page.

In [16]:
report_dropdown = widgets.Dropdown(
    options=history_options,
    description="Model:",
    value=history_options[0][1],
)


def show_classification_report(model_slug):
    experiment = bundle["experiments"][model_slug]
    display(experiment["classification_report"])


display(report_dropdown)
report_output = widgets.interactive_output(
    show_classification_report,
    {"model_slug": report_dropdown},
)
display(report_output)

Dropdown(description='Model:', options=(('ResNet50 (NoAug)', 'resnet50_no_aug'), ('ResNet50 (Aug)', 'resnet50_…

Output()